In [ ]:
# P2PNet for Cell Detection - Imports and Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils import data
import torchvision
from torchvision.models import vgg16_bn, resnet50
import numpy as np
import cv2
import os
import yaml
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from scipy.optimize import linear_sum_assignment
import glob
import json

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 파라미터 로드
with open('utils/args.yaml', errors='ignore') as f:
    params = yaml.safe_load(f)

# 6 클래스 cell type 정의
class_names = {
    0: "Neutrophil",
    1: "Epithelial",
    2: "Lymphocyte",
    3: "Plasma",
    4: "Eosinophil",
    5: "Connective tissue"
}

num_classes = len(class_names)
print(f"Number of classes: {num_classes}")
print(f"Classes: {list(class_names.values())}")

In [ ]:
# P2PNet Model Architecture with Anchor-based Point Prediction
class P2PNet(nn.Module):
    """
    Point-to-Point Network for Cell Detection (Anchor-based)
    - Backbone: VGG16-BN
    - Head: Anchor-based point regression + classification
    - Output: Anchor + offset predictions
    
    🔥 FIX: 각 feature map 위치에 앵커를 두고 offset 예측
    """
    def __init__(self, num_classes=6, backbone='vgg16_bn', row=3, line=3):
        super(P2PNet, self).__init__()
        self.num_classes = num_classes
        self.row = row
        self.line = line
        
        # Backbone - VGG16_BN
        if backbone == 'vgg16_bn':
            vgg = vgg16_bn(pretrained=True)
            self.features = vgg.features
            in_channels = 512
        elif backbone == 'resnet50':
            resnet = resnet50(pretrained=True)
            self.features = nn.Sequential(
                resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
                resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
            )
            in_channels = 2048
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")
        
        # Regression head (offset from anchor)
        self.reg_head = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, row * line * 2, 1)  # offset x, y
        )
        
        # Classification head (num_classes + 1 for background)
        self.cls_head = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, row * line * (num_classes + 1), 1)
        )
        
        # Anchor cache
        self._anchor_cache = {}
        
    def _generate_anchors(self, h_feat, w_feat, device):
        """Generate anchor points for each feature map location"""
        key = (h_feat, w_feat, device)
        if key in self._anchor_cache:
            return self._anchor_cache[key]
        
        # Create grid of anchor centers (normalized 0-1)
        # Each feature map cell center
        y_centers = (torch.arange(h_feat, device=device).float() + 0.5) / h_feat
        x_centers = (torch.arange(w_feat, device=device).float() + 0.5) / w_feat
        
        # Meshgrid
        yy, xx = torch.meshgrid(y_centers, x_centers, indexing='ij')
        
        # Sub-anchors within each cell (row x line grid)
        sub_offsets = []
        for r in range(self.row):
            for l in range(self.line):
                dy = (r + 0.5) / self.row / h_feat - 0.5 / h_feat
                dx = (l + 0.5) / self.line / w_feat - 0.5 / w_feat
                sub_offsets.append((dx, dy))
        
        # Generate all anchors [h, w, row*line, 2]
        anchors = []
        for dx, dy in sub_offsets:
            anchor_x = xx + dx
            anchor_y = yy + dy
            anchors.append(torch.stack([anchor_x, anchor_y], dim=-1))
        
        anchors = torch.stack(anchors, dim=2)  # [h, w, row*line, 2]
        anchors = anchors.reshape(1, h_feat * w_feat * self.row * self.line, 2)
        
        self._anchor_cache[key] = anchors
        return anchors
        
    def forward(self, x):
        batch_size = x.size(0)
        device = x.device
        
        # Backbone
        features = self.features(x)
        
        # Regression head (offset prediction)
        pred_offsets = self.reg_head(features)
        pred_offsets = pred_offsets.permute(0, 2, 3, 1)  # [B, H, W, C]
        
        # Classification head
        pred_logits = self.cls_head(features)
        pred_logits = pred_logits.permute(0, 2, 3, 1)
        
        h_feat, w_feat = pred_offsets.shape[1], pred_offsets.shape[2]
        
        # Reshape
        pred_offsets = pred_offsets.reshape(batch_size, h_feat * w_feat * self.row * self.line, 2)
        pred_logits = pred_logits.reshape(batch_size, h_feat * w_feat * self.row * self.line, self.num_classes + 1)
        
        # Generate anchors
        anchors = self._generate_anchors(h_feat, w_feat, device)
        
        # 🔥 Anchor + scaled offset = final point
        # tanh bounds offset to [-1, 1], scale controls max offset range
        offset_scale = 2.0 / h_feat  # Max offset = 2x cell size (더 넓은 범위)
        pred_offsets = torch.tanh(pred_offsets) * offset_scale
        
        # Final points = anchor + offset  (x, y format)
        pred_points = anchors + pred_offsets
        pred_points = pred_points.clamp(0, 1)
        
        return pred_points, pred_logits


# 모델 초기화 (재생성)
model = P2PNet(num_classes=num_classes, backbone='vgg16_bn', row=2, line=2).to(device)
print(f"\n✅ P2PNet model created with REDUCED PREDICTIONS")
print(f"  Backbone: VGG16-BN")
print(f"  Number of classes: {num_classes} + 1 (background)")
print(f"  Grid: {2}x{2} anchors per feature map cell (감소! 9→4)")
print(f"  🔥 KEY: 예측 수 감소로 false positive 억제!")
print(f"  🔥 16x16 feature map → 16x16x4 = 1,024 predictions (vs 2,304)")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"\n⚠️  PRECISION FIX: 예측 수 56% 감소 (2304→1024)")


In [ ]:
# P2PNet Loss Function with Background Loss (FIXED!)
class P2PNetLoss(nn.Module):
    """
    P2PNet Loss with Background Penalty (DETR-style):
    1. Hungarian Matching based on point distance + class cost
    2. L1 point regression loss (matched only)
    3. Focal loss for classification (ALL predictions!)
       - Matched: target = gt_class
       - Unmatched: target = background_class (num_classes)
    
    🔥 FIX: 매칭되지 않은 예측들을 background class로 학습!
    """
    def __init__(self, num_classes=6, point_loss_coef=2.0, class_loss_coef=1.0, 
                 background_weight=0.02):
        super(P2PNetLoss, self).__init__()
        self.num_classes = num_classes
        self.point_loss_coef = point_loss_coef
        self.class_loss_coef = class_loss_coef
        self.background_weight = background_weight  # Weight for background class
        
    def focal_loss(self, logits, targets, alpha=0.25, gamma=2.0, weight=None):
        """Focal loss for classification with class weights"""
        ce_loss = F.cross_entropy(logits, targets, reduction='none', weight=weight)
        pt = torch.exp(-ce_loss)
        focal_loss = alpha * (1 - pt) ** gamma * ce_loss
        return focal_loss.mean()
    
    def hungarian_matching(self, pred_points, pred_logits, gt_points, gt_classes):
        """
        Hungarian matching based on point distance + class cost
        🔥 Point distance 우선! (cost_weight=3.0)
        """
        if len(gt_points) == 0:
            return torch.tensor([]).long(), torch.tensor([]).long()
        
        # Point distance cost (L1 distance) - 3배 가중치!
        point_cost = torch.cdist(pred_points, gt_points, p=1) * 3.0
        
        # Class cost
        pred_probs = F.softmax(pred_logits, dim=-1)
        class_cost = -pred_probs[:, gt_classes]
        
        # Combined cost matrix (point distance가 더 중요!)
        cost_matrix = point_cost + class_cost
        cost_matrix_np = cost_matrix.detach().cpu().numpy()
        
        # Hungarian matching
        pred_idx, gt_idx = linear_sum_assignment(cost_matrix_np)
        
        return torch.from_numpy(pred_idx).long(), torch.from_numpy(gt_idx).long()
    
    def forward(self, pred_points, pred_logits, targets):
        batch_size = pred_points.size(0)
        num_queries = pred_points.size(1)
        device = pred_points.device
        
        total_point_loss = 0
        total_class_loss = 0
        num_matched = 0
        
        for b in range(batch_size):
            gt_points = targets['points'][b]
            gt_classes = targets['classes'][b].long()
            
            # Filter valid GT points
            valid_mask = (gt_points[:, 0] >= 0) & (gt_points[:, 1] >= 0)
            gt_points = gt_points[valid_mask]
            gt_classes = gt_classes[valid_mask]
            
            # Create target classes for ALL predictions (DETR-style)
            # Default: all predictions are background (class = num_classes)
            target_classes = torch.full((num_queries,), self.num_classes, 
                                       dtype=torch.long, device=device)
            
            if len(gt_points) > 0:
                # Hungarian matching
                pred_idx, gt_idx = self.hungarian_matching(
                    pred_points[b], pred_logits[b], gt_points, gt_classes
                )
                
                if len(pred_idx) > 0:
                    # Point regression loss (matched only)
                    matched_pred_points = pred_points[b][pred_idx]
                    matched_gt_points = gt_points[gt_idx]
                    point_loss = F.l1_loss(matched_pred_points, matched_gt_points, 
                                          reduction='sum')
                    total_point_loss += point_loss
                    num_matched += len(pred_idx)
                    
                    # Set matched predictions to their GT classes
                    target_classes[pred_idx] = gt_classes[gt_idx]
            
            # Classification loss for ALL predictions (matched + unmatched)
            # Matched: target = gt_class
            # Unmatched: target = background_class (num_classes)
            
            # Class weights: background class has lower weight
            class_weights = torch.ones(self.num_classes + 1, device=device)
            class_weights[-1] = self.background_weight  # Background class
            
            # Focal loss with background penalty
            class_loss = self.focal_loss(pred_logits[b], target_classes, weight=class_weights)
            total_class_loss += class_loss
        
        # Average losses
        if num_matched > 0:
            total_point_loss = total_point_loss / num_matched
        else:
            total_point_loss = torch.tensor(0.0, device=device)
        
        total_class_loss = total_class_loss / batch_size
        
        # Weighted sum
        total_loss = (self.point_loss_coef * total_point_loss + 
                     self.class_loss_coef * total_class_loss)
        
        loss_dict = {
            'total': total_loss.item(),
            'point': total_point_loss.item(),
            'class': total_class_loss.item(),
            'num_matched': num_matched
        }
        
        return total_loss, loss_dict


# Loss function 초기화
criterion = P2PNetLoss(
    num_classes=num_classes, 
    point_loss_coef=5.0,  # 🔥 Point localization 최우선
    class_loss_coef=2.0,   # Classification loss
    background_weight=0.01  # 🔥 VERY STRONG! False positive 강력 억제
).to(device)

print(f"\n✅ P2PNet loss function created with STRONG BACKGROUND PENALTY")
print(f"  Point loss coefficient: 5.0 (정확한 위치)")
print(f"  Class loss coefficient: 2.0 (분류)")
print(f"  Background weight: 0.01 (매우 강한 penalty!)")
print(f"  Matching: Hungarian algorithm (point distance 3x weight)")
print()
print(f"  🔥 PRECISION FIX:")
print(f"     - 예측 수 감소 (2304→1024): 구조적 개선")
print(f"     - Background weight 0.01: 빈 공간 예측 강력 억제")
print(f"     - 목표: Precision: 0.3 → 0.6+ 향상")


In [ ]:
# [실행순서 6] Cell 6: Data Loading (same as p2p_train.ipynb)
input_size = 512
label_dir = '../../data/HnE_cell_detect/total_data/labels/'
image_dir = '../../data/HnE_cell_detect/total_data/images/'

label_files = sorted(glob.glob(os.path.join(label_dir, '*.json')))

image_filenames = []
labels = []

print("📂 Loading labels...")
for i in tqdm(range(len(label_files))):
    label_file = label_files[i]
    
    with open(label_file) as f:
        data_json = json.load(f)
    
    img_path = os.path.join(image_dir, data_json['file_name'])
    
    if os.path.exists(img_path):
        image_filenames.append(img_path)
        
        centers = []
        classes = []
        
        # JSON 형식: data_json["cordinates"] = [[class_id, y, x, h, w], ...]
        for coord in data_json["cordinates"]:
            if len(coord) < 5:
                continue
                
            class_id = int(coord[0]) - 1  # HnE 데이터는 1부터 시작하므로 -1
            y = coord[1]
            x = coord[2]
            h = coord[3]
            w = coord[4]
            
            # 너무 큰 박스 제외
            if h > 50 or w > 50:
                continue
            
            # 중심점 좌표 (픽셀 단위)
            centers.append([x+w//2, y+h//2])
            classes.append(class_id)
        
        if len(centers) > 0:
            labels.append({
                'points': np.array(centers, dtype=np.float32),
                'classes': np.array(classes, dtype=np.int64)
            })
        else:
            image_filenames.pop()

print(f"✅ Loaded {len(image_filenames)} images with labels")

print("\n📷 Loading images...")
images = []
for i in tqdm(range(len(image_filenames))):
    image = cv2.imread(image_filenames[i])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    images.append(image)

print(f"✅ Loaded {len(images)} images")
print(f"  Image shape: {images[0].shape}")
print(f"  Label example - Points: {labels[0]['points'].shape}, Classes: {labels[0]['classes'].shape}")

In [ ]:
# Custom Dataset for P2PNet with HnE-specific Augmentation
class P2PNetDataset(data.Dataset):
    def __init__(self, images, labels, img_size=512, augment=False, max_points=2000):
        self.images = images
        self.labels = labels
        self.img_size = img_size
        self.augment = augment
        self.max_points = max_points
        self.n = len(self.images)
    
    def __len__(self):
        return self.n
    
    def apply_color_augmentation(self, image):
        """
        HnE-specific color augmentation for scanner/staining variation
        병리 이미지 스캐너 및 병원별 염색 차이에 대한 증강
        """
        # 1. Brightness adjustment (±20%)
        if random.random() < 0.5:
            brightness_factor = random.uniform(0.8, 1.2)
            image = np.clip(image * brightness_factor, 0, 255).astype(np.uint8)
        
        # 2. Contrast adjustment (±20%)
        if random.random() < 0.5:
            contrast_factor = random.uniform(0.8, 1.2)
            mean = image.mean()
            image = np.clip((image - mean) * contrast_factor + mean, 0, 255).astype(np.uint8)
        
        # 3. Hue shift (HnE stain variation)
        if random.random() < 0.5:
            # Convert to HSV for hue manipulation
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            hue_shift = random.uniform(-10, 10)  # ±10 degrees
            hsv[:, :, 0] = np.clip(hsv[:, :, 0] + hue_shift, 0, 179)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        
        # 4. Saturation adjustment (stain intensity variation)
        if random.random() < 0.5:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            saturation_factor = random.uniform(0.7, 1.3)  # ±30%
            hsv[:, :, 1] = np.clip(hsv[:, :, 1] * saturation_factor, 0, 255)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        
        # 5. Gamma correction (scanner exposure variation)
        if random.random() < 0.3:
            gamma = random.uniform(0.8, 1.2)
            inv_gamma = 1.0 / gamma
            table = np.array([((i / 255.0) ** inv_gamma) * 255 
                            for i in range(256)]).astype(np.uint8)
            image = cv2.LUT(image, table)
        
        # 6. RGB channel shift (scanner color calibration variation)
        if random.random() < 0.3:
            for c in range(3):
                shift = random.uniform(-10, 10)
                image[:, :, c] = np.clip(image[:, :, c].astype(np.float32) + shift, 0, 255).astype(np.uint8)
        
        # 7. Gaussian noise (scanner noise)
        if random.random() < 0.3:
            noise_std = random.uniform(0, 5)
            noise = np.random.normal(0, noise_std, image.shape)
            image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        
        # 8. Gaussian blur (slight focus variation)
        if random.random() < 0.2:
            kernel_size = random.choice([3, 5])
            image = cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
        
        return image
    
    def __getitem__(self, index):
        original_image = self.images[index].copy()  # Keep original image reference
        points = self.labels[index]['points'].copy()
        classes = self.labels[index]['classes'].copy()
        crop_points = []
        crop_classes = []
        while(len(crop_points)<1):
            image, h1, w1 = self.crop_padding_image(original_image)  # Always crop from original!
            crop_points = []
            crop_classes = []
            for i in range(len(classes)):
                x = points[i][0]
                y = points[i][1]
                # 중심점 계산
                center_x = x
                center_y = y
                
                # 크롭된 영역 내에 중심점이 있는지 확인
                if (center_x >= w1 and center_y >= h1 and 
                    center_x <= w1 + self.img_size and 
                    center_y <= h1 + self.img_size):
                    # 절대 픽셀 좌표로 변환 (0~512 범위)
                    abs_x = center_x - w1
                    abs_y = center_y - h1
                    crop_points.append([float(abs_x), float(abs_y)])
                    crop_classes.append(classes[i])  # 클래스 인덱스 (0-based)
        points = np.array(crop_points, dtype=np.float32)
        classes = np.array(crop_classes, dtype=np.int64)
        # Geometric augmentation (preserves spatial relationships)
        if self.augment:
            # Horizontal flip
            if random.random() < 0.5:
                image = np.fliplr(image).copy()
                points[:, 0] = self.img_size - points[:, 0]
            
            # Vertical flip
            if random.random() < 0.5:
                image = np.flipud(image).copy()
                points[:, 1] = self.img_size - points[:, 1]
            
            # 90-degree rotation (with point adjustment)
            if random.random() < 0.3:
                k = random.randint(1, 3)  # 90, 180, or 270 degrees
                image = np.rot90(image, k).copy()
                for _ in range(k):
                    # Rotate points 90 degrees clockwise
                    new_points = points.copy()
                    new_points[:, 0] = points[:, 1]
                    new_points[:, 1] = self.img_size - points[:, 0]
                    points = new_points
            
            # Color augmentation (HnE-specific)
            image = self.apply_color_augmentation(image)
        
        # Normalize point coordinates to [0, 1]
        points[:, 0] = points[:, 0] / self.img_size
        points[:, 1] = points[:, 1] / self.img_size
        
        # Normalize image to [0, 1]
        image = image.astype(np.float32) / 255.0
        image = image.transpose((2, 0, 1))  # HWC -> CHW
        
        # Pad or truncate points to max_points
        num_points = len(points)
        if num_points < self.max_points:
            padded_points = np.full((self.max_points, 2), -1.0, dtype=np.float32)
            padded_classes = np.full((self.max_points,), -1, dtype=np.int64)
            
            padded_points[:num_points] = points
            padded_classes[:num_points] = classes
            
            points = padded_points
            classes = padded_classes
        else:
            points = points[:self.max_points]
            classes = classes[:self.max_points]
        
        return (torch.from_numpy(image).float(),
                torch.from_numpy(points).float(),
                torch.from_numpy(classes).long())
        
    def crop_padding_image(self,image):
        image = image.copy()
        h, w = image.shape[:2]
        r = self.img_size / min(h, w)
        
        # 이미지가 input_size보다 큰 경우 랜덤 크롭
        if r < 1:
            max_h = max(0, h - self.img_size)
            max_w = max(0, w - self.img_size)
            h1 = random.randint(0, max_h) if max_h > 0 else 0
            w1 = random.randint(0, max_w) if max_w > 0 else 0
            image = image[h1:h1 + self.img_size, w1:w1 + self.img_size]
        else:
            # 이미지가 input_size보다 작은 경우 패딩
            h1 = 0
            w1 = 0
            pad_image = np.ones((self.img_size, self.img_size, 3), dtype=np.uint8) * 255
            pad_image[:min(h, self.img_size), :min(w, self.img_size), :] = image[:min(h, self.img_size), :min(w, self.img_size), :]
            image = pad_image
        return image,h1,w1
def collate_fn_p2pnet(batch):
    images, points, classes = zip(*batch)
    
    images = torch.stack(images, dim=0)
    points = torch.stack(points, dim=0)
    classes = torch.stack(classes, dim=0)
    
    targets = {
        'points': points,
        'classes': classes
    }
    
    return images, targets


# Train/Val split
train_images, val_images, train_labels, val_labels = train_test_split(
    images, labels, test_size=0.1, random_state=242, shuffle=True
)

train_dataset = P2PNetDataset(train_images, train_labels, augment=True, max_points=2000)
val_dataset = P2PNetDataset(val_images, val_labels, augment=False, max_points=2000)

print(f"\n📊 Dataset split:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Max points per image: 2000")

print(f"\n🎨 HnE-specific augmentations (Training only):")
print(f"  ✅ Geometric: Horizontal/Vertical flip, 90° rotation")
print(f"  ✅ Brightness: ±20% (scanner exposure variation)")
print(f"  ✅ Contrast: ±20% (scanner contrast variation)")
print(f"  ✅ Hue shift: ±10° (stain color variation)")
print(f"  ✅ Saturation: ±30% (stain intensity variation)")
print(f"  ✅ Gamma correction: 0.8-1.2 (scanner gamma variation)")
print(f"  ✅ RGB channel shift: ±10 (color calibration variation)")
print(f"  ✅ Gaussian noise: 0-5 std (scanner noise)")
print(f"  ✅ Gaussian blur: 3-5 kernel (focus variation)")

batch_size = 8
train_loader = data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=4, collate_fn=collate_fn_p2pnet, pin_memory=True
)
val_loader = data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, collate_fn=collate_fn_p2pnet, pin_memory=True
)

print(f"\n📦 Dataloaders created:")
print(f"  Batch size: {batch_size}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")


In [ ]:
# Visualization function
def visualize_p2pnet_sample(dataset, index=0):
    image_tensor, points_tensor, classes_tensor = dataset[index]
    
    image = image_tensor.numpy().transpose(1, 2, 0)
    points = points_tensor.numpy()
    classes = classes_tensor.numpy()
    
    valid_mask = (points[:, 0] >= 0) & (points[:, 1] >= 0)
    points = points[valid_mask]
    classes = classes[valid_mask]
    
    h, w = image.shape[:2]
    points_pixel = points.copy()
    points_pixel[:, 0] *= w
    points_pixel[:, 1] *= h
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(image)
    
    colors = ['red', 'green', 'yellow', 'magenta', 'dodgerblue', 'orange', 'gray']
    
    for i in range(len(points_pixel)):
        x, y = points_pixel[i]
        class_id = int(classes[i])
        color = colors[class_id] if class_id < len(colors) else 'white'
        
        circle = plt.Circle((x, y), 3, color=color, fill=True, alpha=0.7)
        ax.add_patch(circle)
    
    ax.set_title(f'Sample {index} - Total points: {len(points)}', fontsize=14, fontweight='bold')
    ax.axis('off')
    
    legend_elements = [
        patches.Patch(color=colors[i], label=f'{class_names[i]}: {sum(classes==i)}')
        for i in range(num_classes) if sum(classes==i) > 0
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Sample {index} statistics:")
    print(f"  Total points: {len(points)}")
    for i in range(num_classes):
        count = sum(classes == i)
        if count > 0:
            print(f"  {class_names[i]}: {count}")


def visualize_augmentation_effects(dataset, index=5, num_augmentations=6):
    """
    Visualize the effect of HnE-specific augmentations
    증강 효과 시각화
    """
    # Get original image (without augmentation)
    original_image = dataset.images[index].copy()
    points = dataset.labels[index]['points'].copy()
    classes = dataset.labels[index]['classes'].copy()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    colors_map = ['red', 'green', 'yellow', 'magenta', 'dodgerblue', 'orange']
    
    # Original image
    axes[0].imshow(original_image)
    for i in range(len(points)):
        x, y = points[i]
        class_id = int(classes[i])
        color = colors_map[class_id] if class_id < len(colors_map) else 'white'
        circle = plt.Circle((x, y), 4, color=color, fill=True, alpha=0.7)
        axes[0].add_patch(circle)
    axes[0].set_title('Original Image', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Augmented versions
    for aug_idx in range(1, num_augmentations):
        # Get augmented sample
        image_tensor, points_tensor, classes_tensor = dataset[index]
        
        image = image_tensor.numpy().transpose(1, 2, 0)
        aug_points = points_tensor.numpy()
        aug_classes = classes_tensor.numpy()
        
        valid_mask = (aug_points[:, 0] >= 0) & (aug_points[:, 1] >= 0)
        aug_points = aug_points[valid_mask]
        aug_classes = aug_classes[valid_mask]
        
        h, w = image.shape[:2]
        aug_points_pixel = aug_points.copy()
        aug_points_pixel[:, 0] *= w
        aug_points_pixel[:, 1] *= h
        
        axes[aug_idx].imshow(image)
        for i in range(len(aug_points_pixel)):
            x, y = aug_points_pixel[i]
            class_id = int(aug_classes[i])
            color = colors_map[class_id] if class_id < len(colors_map) else 'white'
            circle = plt.Circle((x, y), 4, color=color, fill=True, alpha=0.7)
            axes[aug_idx].add_patch(circle)
        axes[aug_idx].set_title(f'Augmented {aug_idx}', fontsize=12, fontweight='bold')
        axes[aug_idx].axis('off')
    
    plt.suptitle('HnE Augmentation Examples (Scanner/Staining Variation)', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n🎨 각 증강 이미지는 다른 스캐너나 병원의 염색 변화를 시뮬레이션합니다:")
    print("  - 색상, 명도, 대비, 채도의 변화")
    print("  - 스캐너별 색보정 차이")
    print("  - 염색 강도 및 색조 변화")


print("🖼️ Visualizing training sample...")
visualize_p2pnet_sample(train_dataset, index=5)

print("\n🎨 Visualizing augmentation effects...")
visualize_augmentation_effects(train_dataset, index=5, num_augmentations=6)


In [ ]:
# Training Setup
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

epochs = 300
save_dir = '../../model/HnE_cell_detection/p2pnet/'
os.makedirs(save_dir, exist_ok=True)

train_losses = []
val_point_errors = []
val_class_accs = []
val_recalls = []
val_precisions = []
best_val_point_error = float('inf')

print(f"\n🚀 Training setup (Official P2PNet):")
print(f"  Epochs: {epochs}")
print(f"  Optimizer: AdamW (lr=1e-4, wd=1e-4)")
print(f"  Scheduler: CosineAnnealingWarmRestarts")
print(f"  Save directory: {save_dir}")
print(f"  Removed: Objectness tracking")

# Optional: Load previous checkpoint
# checkpoint_path = f'{save_dir}last_model.pt'
# if os.path.exists(checkpoint_path):
#     checkpoint = torch.load(checkpoint_path, map_location=device)
#     model.load_state_dict(checkpoint['model_state_dict'])
#     print(f"✅ Loaded checkpoint from epoch {checkpoint['epoch']+1}")


In [ ]:
# Evaluation and Visualization (Updated for Official P2PNet)
def nms_points(points, scores, classes, nms_threshold=10.0):
    """Non-Maximum Suppression for point predictions (강화된 NMS)"""
    if len(points) == 0:
        return np.array([])
    
    points_pixel = points * 512
    sorted_indices = np.argsort(-scores)
    
    keep = []
    while len(sorted_indices) > 0:
        current_idx = sorted_indices[0]
        keep.append(current_idx)
        
        if len(sorted_indices) == 1:
            break
        
        current_point = points_pixel[current_idx]
        remaining_points = points_pixel[sorted_indices[1:]]
        distances = np.sqrt(np.sum((remaining_points - current_point) ** 2, axis=1))
        
        current_class = classes[current_idx]
        remaining_classes = classes[sorted_indices[1:]]
        same_class = (remaining_classes == current_class)
        
        keep_mask = (distances > nms_threshold) | (~same_class)
        sorted_indices = sorted_indices[1:][keep_mask]
    
    return np.array(keep)


def visualize_predictions(model, dataset, index=0, conf_threshold=0.75, nms_threshold=10.0, 
                         save_path=None, show=True):
    """Visualize predictions (강화된 NMS로 false positive 억제)"""
    model.eval()
    
    image_tensor, gt_points_tensor, gt_classes_tensor = dataset[index]
    
    valid_mask = (gt_points_tensor[:, 0] >= 0) & (gt_points_tensor[:, 1] >= 0)
    gt_points = gt_points_tensor[valid_mask].numpy()
    gt_classes = gt_classes_tensor[valid_mask].numpy()
    
    with torch.no_grad():
        image_batch = image_tensor.unsqueeze(0).to(device)
        pred_points, pred_logits = model(image_batch)
        
        # Filter out background class (last class)
        pred_probs = F.softmax(pred_logits[0], dim=-1)
        foreground_probs = pred_probs[:, :-1]  # Exclude background class
        
        pred_points = pred_points[0].cpu().numpy()
        pred_scores, pred_classes = foreground_probs.max(dim=-1)
        pred_scores = pred_scores.cpu().numpy()
        pred_classes = pred_classes.cpu().numpy()
    
    if show:
        print(f"  Total predictions: {len(pred_points)}")
    
    # Filter by confidence threshold (NO objectness)
    conf_mask = pred_scores > conf_threshold
    pred_points = pred_points[conf_mask]
    pred_classes = pred_classes[conf_mask]
    pred_scores = pred_scores[conf_mask]
    
    if show:
        print(f"  After confidence (>{conf_threshold}): {len(pred_points)}")
    
    # Apply NMS
    if len(pred_points) > 0:
        keep_indices = nms_points(pred_points, pred_scores, pred_classes, nms_threshold=nms_threshold)
        pred_points = pred_points[keep_indices]
        pred_classes = pred_classes[keep_indices]
        pred_scores = pred_scores[keep_indices]
        
        if show:
            print(f"  After NMS ({nms_threshold}px): {len(pred_points)}")
    
    image = image_tensor.numpy().transpose(1, 2, 0)
    h, w = image.shape[:2]
    
    gt_points_pixel = gt_points.copy()
    gt_points_pixel[:, 0] *= w
    gt_points_pixel[:, 1] *= h
    
    pred_points_pixel = pred_points.copy()
    pred_points_pixel[:, 0] *= w
    pred_points_pixel[:, 1] *= h
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    colors = ['red', 'green', 'yellow', 'magenta', 'dodgerblue', 'orange']
    
    # Ground Truth
    axes[0].imshow(image)
    for i in range(len(gt_points_pixel)):
        x, y = gt_points_pixel[i]
        class_id = int(gt_classes[i])
        color = colors[class_id] if class_id < len(colors) else 'white'
        circle = plt.Circle((x, y), 4, color=color, fill=True, alpha=0.8)
        axes[0].add_patch(circle)
    axes[0].set_title(f'Ground Truth ({len(gt_points_pixel)} points)', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # Predictions
    axes[1].imshow(image)
    for i in range(len(pred_points_pixel)):
        x, y = pred_points_pixel[i]
        class_id = int(pred_classes[i])
        color = colors[class_id] if class_id < len(colors) else 'white'
        circle = plt.Circle((x, y), 4, color=color, fill=True, alpha=0.8)
        axes[1].add_patch(circle)
    
    axes[1].set_title(f'Predictions ({len(pred_points_pixel)} points)', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    # Legend
    legend_elements = [
        patches.Patch(color=colors[i], label=class_names[i])
        for i in range(num_classes)
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=num_classes,
              bbox_to_anchor=(0.5, -0.05), fontsize=12)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.1)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        if show:
            print(f"✅ Saved visualization to: {save_path}")
    
    if show:
        plt.show()
        print(f"\n📊 Sample {index} statistics:")
        print(f"  Ground Truth: {len(gt_points_pixel)} points")
        print(f"  Predictions: {len(pred_points_pixel)} points")
    else:
        plt.close()





In [ ]:
# Training Loop (Updated for Official P2PNet)
print("\n" + "="*80)
print("🚀 Starting P2PNet Training (Official Structure)")
print("="*80)

for epoch in range(epochs):
    # TRAINING
    model.train()
    train_loss = 0
    train_point_loss = 0
    train_class_loss = 0
    train_matched = 0
    
    train_pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                     desc=f'Epoch {epoch+1}/{epochs} [Train]')
    
    for batch_idx, (images, targets) in train_pbar:
        images = images.to(device)
        targets = {k: v.to(device) for k, v in targets.items()}
        
        # Forward pass (NO pred_obj anymore)
        pred_points, pred_logits = model(images)
        
        if epoch == 0 and batch_idx == 0:
            print(f"\n📊 First batch info:")
            print(f"  Input shape: {images.shape}")
            print(f"  Pred points: {pred_points.shape}")
            print(f"  Pred logits: {pred_logits.shape}")
            print(f"  GT points: {targets['points'].shape}")
            
            valid_mask = (targets['points'][0, :, 0] >= 0) & (targets['points'][0, :, 1] >= 0)
            num_valid = valid_mask.sum().item()
            print(f"  Valid GT points: {num_valid}")
        
        # Compute loss
        loss, loss_dict = criterion(pred_points, pred_logits, targets)
        
        if epoch == 0 and batch_idx == 0:
            print(f"\n💰 First batch loss:")
            print(f"  Total: {loss_dict['total']:.4f}")
            print(f"  Point: {loss_dict['point']:.4f} (coef=0.0002)")
            print(f"  Class: {loss_dict['class']:.4f} (coef=1.0)")
            print(f"  Matched: {loss_dict['num_matched']}\n")
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss_dict['total']
        train_point_loss += loss_dict['point']
        train_class_loss += loss_dict['class']
        train_matched += loss_dict['num_matched']
        
        memory = f'{torch.cuda.memory_reserved() / 1E9:.2f}G'
        train_pbar.set_postfix({
            'loss': f"{loss_dict['total']:.4f}",
            'pt': f"{loss_dict['point']:.4f}",
            'cls': f"{loss_dict['class']:.4f}",
            'mem': memory
        })
    
    num_batches = len(train_loader)
    avg_train_loss = train_loss / num_batches
    avg_train_point_loss = train_point_loss / num_batches
    avg_train_class_loss = train_class_loss / num_batches
    avg_train_matched = train_matched / num_batches
    train_losses.append(avg_train_loss)
    
    # VALIDATION
    model.eval()
    val_point_error = 0
    val_class_correct = 0
    val_total = 0
    val_recall_sum = 0
    val_precision_sum = 0
    val_samples = 0
    
    val_pbar = tqdm(enumerate(val_loader), total=len(val_loader),
                   desc=f'Epoch {epoch+1}/{epochs} [Val]')
    
    with torch.no_grad():
        for batch_idx, (images, targets) in val_pbar:
            images = images.to(device)
            targets = {k: v.to(device) for k, v in targets.items()}
            
            # Forward pass
            pred_points, pred_logits = model(images)
            
            batch_size = images.size(0)
            for b in range(batch_size):
                gt_points = targets['points'][b]
                gt_classes = targets['classes'][b].long()
                
                valid_mask = (gt_points[:, 0] >= 0) & (gt_points[:, 1] >= 0)
                gt_points = gt_points[valid_mask]
                gt_classes = gt_classes[valid_mask]
                
                if len(gt_points) == 0:
                    continue
                
                # Filter out background class predictions
                pred_probs_bg = F.softmax(pred_logits[b], dim=-1)
                # Get max prob excluding background (last class)
                foreground_probs = pred_probs_bg[:, :-1]  # Exclude background
                max_probs, pred_classes = foreground_probs.max(dim=-1)
                
                # Filter by confidence threshold
                conf_threshold = 0.7  # 🔥 HIGH! Precision 최우선
                conf_mask = max_probs > conf_threshold
                
                if conf_mask.sum() == 0:
                    val_recall_sum += 0
                    val_precision_sum += 0
                    val_samples += 1
                    continue
                
                filtered_pred_points = pred_points[b][conf_mask]
                filtered_pred_logits = pred_logits[b][conf_mask]
                
                # Hungarian matching
                point_cost = torch.cdist(filtered_pred_points, gt_points, p=1)
                pred_probs_match = F.softmax(filtered_pred_logits, dim=-1)
                class_cost = -pred_probs_match[:, gt_classes]
                cost_matrix = (point_cost + class_cost).detach().cpu().numpy()
                
                pred_idx, gt_idx = linear_sum_assignment(cost_matrix)
                
                # Calculate metrics
                matched_pred_points = filtered_pred_points[pred_idx]
                matched_gt_points = gt_points[gt_idx]
                point_error = torch.abs(matched_pred_points - matched_gt_points).mean()
                val_point_error += point_error.item()
                
                matched_pred_classes = filtered_pred_logits[pred_idx].argmax(dim=-1)
                matched_gt_classes = gt_classes[gt_idx]
                correct = (matched_pred_classes == matched_gt_classes).sum().item()
                val_class_correct += correct
                val_total += len(gt_idx)
                
                recall = len(gt_idx) / len(gt_points)
                precision = len(pred_idx) / conf_mask.sum().item() if conf_mask.sum() > 0 else 0
                val_recall_sum += recall
                val_precision_sum += precision
                val_samples += 1
    
    avg_val_point_error = val_point_error / val_samples if val_samples > 0 else 0
    avg_val_class_acc = val_class_correct / val_total if val_total > 0 else 0
    avg_val_recall = val_recall_sum / val_samples if val_samples > 0 else 0
    avg_val_precision = val_precision_sum / val_samples if val_samples > 0 else 0
    avg_val_f1 = 2 * avg_val_precision * avg_val_recall / (avg_val_precision + avg_val_recall + 1e-8)
    
    val_point_errors.append(avg_val_point_error)
    val_class_accs.append(avg_val_class_acc)
    val_recalls.append(avg_val_recall)
    val_precisions.append(avg_val_precision)
    
    scheduler.step()
    
    # LOGGING
    print(f"\n{'='*80}")
    print(f"Epoch {epoch+1}/{epochs} Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f} (Pt: {avg_train_point_loss:.4f}, Cls: {avg_train_class_loss:.4f})")
    print(f"  Train Matched: {avg_train_matched:.1f} points/batch")
    print(f"  Val Point Error: {avg_val_point_error:.4f}")
    print(f"  Val Class Acc: {avg_val_class_acc:.4f}")
    print(f"  Val Recall: {avg_val_recall:.4f}")
    print(f"  Val Precision: {avg_val_precision:.4f}")
    print(f"  Val F1: {avg_val_f1:.4f}")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"{'='*80}\n")
    
    # SAVE CHECKPOINT
    if avg_val_point_error < best_val_point_error:
        best_val_point_error = avg_val_point_error
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': avg_train_loss,
            'val_point_error': avg_val_point_error,
            'val_class_acc': avg_val_class_acc,
            'best_val_point_error': best_val_point_error
        }
        torch.save(checkpoint, os.path.join(save_dir, 'best_model.pt'))
        print(f"🎉 New best model saved! Point Error: {avg_val_point_error:.4f}\n")
    
    last_checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': avg_train_loss,
        'val_point_error': avg_val_point_error,
        'val_class_acc': avg_val_class_acc
    }
    torch.save(last_checkpoint, os.path.join(save_dir, 'last_model.pt'))
    
    # SAVE VISUALIZATION IMAGES (Every 10 epochs)
    if (epoch + 1) % 10 == 0:
        vis_save_path = os.path.join(save_dir, f'predictions_epoch_{epoch+1}.png')
        vis_idx = random.randint(0, len(val_dataset)-1)
        visualize_predictions(model, val_dataset, index=vis_idx, 
                            conf_threshold=0.75, nms_threshold=10.0,
                            save_path=vis_save_path, show=False)
        print(f"📸 Saved visualization: predictions_epoch_{epoch+1}.png\n")
    
    # PLOT PROGRESS (Every 100 epochs)
    if (epoch + 1) % 100 == 0:
        fig, axes = plt.subplots(2, 2, figsize=(18, 12))
        
        axes[0,0].plot(train_losses, 'b-')
        axes[0,0].set_title('Training Loss')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].grid(True)
        
        axes[0,1].plot(val_point_errors, 'r-')
        axes[0,1].set_title('Val Point Error')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].grid(True)
        
        axes[1,0].plot(val_class_accs, 'g-')
        axes[1,0].set_title('Val Class Accuracy')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].grid(True)
        
        axes[1,1].plot(val_recalls, 'c-', label='Recall')
        axes[1,1].plot(val_precisions, 'm-', label='Precision')
        axes[1,1].set_title('Val Recall & Precision')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].legend()
        axes[1,1].grid(True)
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f'training_progress_epoch_{epoch+1}.png'), dpi=150)
        plt.close()
        print(f"📊 Saved progress plot: training_progress_epoch_{epoch+1}.png\n")

print("\n" + "="*80)
print("🎯 Training Complete!")
print(f"  Best Val Point Error: {best_val_point_error:.4f}")
print(f"  Models saved to: {save_dir}")

print("="*80)

In [ ]:
targets